# זרימת עבודה עם אדם בלולאה במסגרת Microsoft Agent Framework

## 🎯 מטרות למידה

בפנקס זה תלמדו כיצד להטמיע זרימות עבודה של **אדם בלולאה** באמצעות `RequestInfoExecutor` של Microsoft Agent Framework. תבנית חזקה זו מאפשרת לכם להשהות את זרימות העבודה של הבינה המלאכותית כדי לאסוף קלט אנושי, מה שהופך את הסוכנים שלכם לאינטראקטיביים ומעניק לאנשים שליטה על החלטות קריטיות.

## 🔄 מה זה אדם בלולאה?

**אדם בלולאה (HITL)** הוא תבנית עיצוב שבה סוכני AI מפסיקים את ההרצה כדי לבקש קלט אנושי לפני ההמשך. זה חיוני עבור:

- ✅ **החלטות קריטיות** - לקבל אישור אנושי לפני ביצוע פעולות חשובות
- ✅ **מצבים מעורפלים** - לאפשר לאנשים להבהיר כאשר AI אינו בטוח
- ✅ **העדפות משתמשים** - לבקש מהמשתמשים לבחור בין אפשרויות שונות
- ✅ **ציות ובטיחות** - להבטיח פיקוח אנושי על פעולות מפוקחות
- ✅ **חוויות אינטראקטיביות** - לבנות סוכנים שיחותיים שמגיבים לקלט המשתמש

## 🏗️ כיצד זה עובד במסגרת Microsoft Agent Framework

המסגרת מספקת שלושה רכיבים מרכזיים ל-HITL:

1. **`RequestInfoExecutor`** - מבצע מיוחד שמפסיק את זרימת העבודה ומשדר `RequestInfoEvent`
2. **`RequestInfoMessage`** - מחלקת בסיס להטענות בקשות מוקלדות שנשלחות לבני אדם
3. **`RequestResponse`** - מקשר תגובות אנושיות עם הבקשות המקוריות באמצעות `request_id`

**תבנית זרימת עבודה:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 הדוגמה שלנו: הזמנת מלון עם אישור משתמש

נרחיב את זרימת העבודה המותנית על ידי הוספת אישור אנושי **לפני** הצעת יעדים חלופיים:

1. המשתמש מבקש יעד (למשל, "פריז")
2. `availability_agent` בודק אם יש חדרים זמינים
3. **אם אין חדרים** → `confirmation_agent` שואל "האם תרצה לראות אפשרויות חלופיות?"
4. זרימת העבודה **עוצרת** באמצעות `RequestInfoExecutor`
5. **האדם משיב** "כן" או "לא" דרך קלט בקונסולה
6. `decision_manager` מנתב לפי התגובה:
   - **כן** → הצגת יעדים חלופיים
   - **לא** → ביטול בקשת ההזמנה
7. הצגת התוצאה הסופית

זה מדגים כיצד להעניק למשתמשים שליטה על ההצעות של הסוכן!

---

בואו נתחיל! 🚀


## שלב 1: ייבוא ספריות נדרשות

אנו מייבאים את רכיבי מסגרת הסוכן הסטנדרטית בנוסף ל**מחלקות ספציפיות לאינטראקציה עם אדם במחזור**:
- `RequestInfoExecutor` - מבצע שמפסיק את זרימת העבודה לקבלת קלט מהאדם
- `RequestInfoEvent` - אירוע שנפלט כאשר מתבקש קלט מהאדם
- `RequestInfoMessage` - מחלקה בסיסית למשא סרגל בקשות מסוג מסוים
- `RequestResponse` - מקשר בין תגובות האדם לבקשות
- `WorkflowOutputEvent` - אירוע לזיהוי פלט של זרימת העבודה


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## שלב 2: הגדרת מודלים של Pydantic לפלטים מובנים

מודלים אלו מגדירים את **הסקמה** שהסוכנים יחזירו. אנו שומרים את כל המודלים מזרימת העבודה המותנית ומוסיפים:

**חדש למעורבות אדם בשרשרת:**
- `HumanFeedbackRequest` - תת-מחלקה של `RequestInfoMessage` המגדירה את המטען הנשלח לבני אדם
  - מכיל את `prompt` (שאלה לשאול) ו-`destination` (הקשר לגבי העיר שאינה זמינה)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## שלב 3: יצירת כלי הזמנת המלון

אותו כלי כמו בזרימת העבודה המותנית - בודק אם יש חדרים זמינים ביעד.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## שלב 4: הגדרת פונקציות תנאי לניתוב

אנו צריכים **ארבע פונקציות תנאי** עבור זרימת העבודה של האדם במחזור:

**מזרימת העבודה התנאיית:**
1. `has_availability_condition` - מנווט כאשר יש זמינות במלונות
2. `no_availability_condition` - מנווט כאשר אין זמינות במלונות

**חדש עבור אדם במחזור:**
3. `user_wants_alternatives_condition` - מנווט כאשר המשתמש אומר "כן" לאלטרנטיבות
4. `user_declines_alternatives_condition` - מנווט כאשר המשתמש אומר "לא" לאלטרנטיבות


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## שלב 5: צור את מנהל ההרצה Decision Manager

זהו **המרכז של תבנית האדם-במעגל**! ה-`DecisionManager` הוא `Executor` מותאם אישית ש-

1. **מקבל משוב מהאדם** דרך אובייקטים מסוג `RequestResponse`
2. **מעבד את החלטת המשתמש** (כן/לא)
3. **מנתב את תהליך העבודה** על ידי שליחת הודעות לסוכנים מתאימים

תכונות מרכזיות:
- משתמש במעטר `@handler` כדי לחשוף שיטות כשלבי תהליך עבודה
- מקבל `RequestResponse[HumanFeedbackRequest, str]` המכיל גם את הבקשה המקורית וגם את תשובת המשתמש
- מייצר הודעות פשוטות של "כן" או "לא" שמפעילות את פונקציות התנאי שלנו


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## שלב 6: צור מבצע תצוגה מותאם אישית

אותו מבצע תצוגה מהזרימה התנאי - מניב תוצאות סופיות כתוצאת הזרימה.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## שלב 7: טעינת משתני סביבה

הגדר את לקוח ה-LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## שלב 8: יצירת סוכני AI ומבצעים

אנו יוצרים **שישה רכיבי זרימת עבודה**:

**סוכנים (עטופים ב-AgentExecutor):**
1. **availability_agent** - בודק זמינות מלון באמצעות הכלי
2. **confirmation_agent** - 🆕 מכין בקשת אישור אנושית
3. **alternative_agent** - מציע ערים חלופיות (כאשר המשתמש אומר כן)
4. **booking_agent** - מעודד הזמנה (כאשר חדרים זמינים)
5. **cancellation_agent** - 🆕 מטפל בהודעת ביטול (כאשר המשתמש אומר לא)

**מבצעים מיוחדים:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` שמעצור את זרימת העבודה לקבלת קלט אנושי
7. **decision_manager** - 🆕 מבצע מותאם שמנתב לפי תגובת האדם (כבר מוגדר למעלה)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## שלב 9: בניית זרימת העבודה עם אדם בתהליך

עכשיו אנו בונים את גרף זרימת העבודה עם **ניתוב מותנה** הכולל את המסלול עם אדם בתהליך:

**מבנה זרימת העבודה:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**קצוות מרכזיים:**
- `availability_agent → confirmation_agent` (כאשר אין חדרים)
- `confirmation_agent → prepare_human_request` (שינוי סוג)
- `prepare_human_request → request_info_executor` (הפסקה עבור האדם)
- `request_info_executor → decision_manager` (תמיד - מספק RequestResponse)
- `decision_manager → alternative_agent` (כאשר המשתמש אומר "כן")
- `decision_manager → cancellation_agent` (כאשר המשתמש אומר "לא")
- `availability_agent → booking_agent` (כאשר חדרים זמינים)
- כל המסלולים מסתיימים ב- `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## שלב 10: הפעלת מקרה מבחן 1 - עיר ללא זמינות (פריז עם אישור אנושי)

מבחן זה מדגים את **מחזור המעורבות האנושי המלא**:

1. בקשת מלון בפריז
2. availability_agent בודק → אין חדרים
3. confirmation_agent יוצר שאלה המופנית לאדם
4. request_info_executor **עוצר את זרימת העבודה** ומשדר `RequestInfoEvent`
5. **היישום מזהה את האירוע ומבקש מהמשתמש במסוף**
6. המשתמש מקליד "yes" או "no"
7. היישום שולח תגובה דרך `send_responses_streaming()`
8. decision_manager מנתב בהתאם לתגובה
9. התוצאה הסופית מוצגת

**תבנית מפתח:**
- השתמש ב- `workflow.run_stream()` לאיטרציה הראשונה
- השתמש ב- `workflow.send_responses_streaming(pending_responses)` לאיטרציות הבאות
- האזן ל- `RequestInfoEvent` לזיהוי מתי נדרש קלט אנושי
- האזן ל- `WorkflowOutputEvent` כדי ללכוד תוצאות סופיות


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## שלב 11: הרץ מקרה מבחן 2 - עיר עם זמינות (סטוקהולם - ללא צורך בקלט אנושי)

מבחן זה מדגים את **הדרך הישירה** כאשר חדרים זמינים:

1. בקשת מלון בסטוקהולם
2. availability_agent בודק → חדרים זמינים ✅
3. booking_agent מציע הזמנה
4. display_result מציג אישור
5. **אינו נדרש קלט אנושי!**

תהליך העבודה עוקף לחלוטין את המסלול האנושי כשחדרים זמינים.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## נקודות מפתח ופרקטיקות מומלצות לאדם בלולאה

### ✅ מה שלמדת:

#### 1. **תבנית RequestInfoExecutor**
תבנית האדם בלולאה במסגרת Microsoft Agent משתמשת בשלושה רכיבים מרכזיים:
- `RequestInfoExecutor` - עוצר את תהליך העבודה ומשדר אירועים
- `RequestInfoMessage` - מחלקת בסיס להטענות בקשתיות ממוספרות (ירש ממנה!)
- `RequestResponse` - מקשרת תגובות אדם עם הבקשות המקוריות

**הבנה קריטית:**
- `RequestInfoExecutor` אינו אוסף את הקלט בעצמו - הוא רק עוצר את תהליך העבודה
- הקוד של האפליקציה שלך חייב להאזין ל-`RequestInfoEvent` ולאסוף קלט
- עליך לקרוא ל-`send_responses_streaming()` עם מילון שממפה `request_id` לתשובת המשתמש

#### 2. **תבנית ביצוע בזרימה**
```python
# איטרציה ראשונה
stream = workflow.run_stream(initial_request)

# איטרציות הבאות (לאחר קלט אנושי)
stream = workflow.send_responses_streaming(pending_responses)

# תמיד לעבד אירועים
events = [event async for event in stream]
```

#### 3. **ארכיטקטורה מונעת אירועים**
האזן לאירועים ספציפיים לשליטה בתהליך העבודה:
- `RequestInfoEvent` - נדרש קלט אדם (תהליך העבודה נעצר)
- `WorkflowOutputEvent` - התוצאה הסופית זמינה (תהליך העבודה הושלם)
- `WorkflowStatusEvent` - שינויים במצב (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, ועוד)

#### 4. **מבצעים מותאמים אישית עם @handler**
ה-`DecisionManager` מדגים כיצד ליצור מבצעים ש:
- משתמשים בדקורטור `@handler` לחשיפת שיטות כשלבי תהליך עבודה
- מקבלים הודעות ממוספרות (לדוגמה, `RequestResponse[HumanFeedbackRequest, str]`)
- מנתבים תהליך עבודה על ידי שליחת הודעות למבצעים אחרים
- ניגשים להקשר דרך `WorkflowContext`

#### 5. **ניתוב מותנה עם החלטות אדם**
ניתן ליצור פונקציות תנאי שמעריכות תגובות אדם:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 יישומים במציאות:

1. **תהליכי אישור**
   - קבלת אישור מנהל לפני טיפול בדוח הוצאות
   - דרישת סקירה אנושית לפני שליחת מיילים אוטומטיים
   - אישור עסקאות בעלות ערך גבוה לפני ביצוע

2. **מודרציה של תוכן**
   - סימון תוכן חשוד לסקירה אנושית
   - בקשת החלטה סופית במקרים שוליים
   - העברה לאדם כאשר ביטחון ה-AI נמוך

3. **שירות לקוחות**
   - לאפשר ל-AI לטפל בשאלות שגרתיות אוטומטית
   - העברת נושאים מורכבים לסוכני שירות אנושיים
   - לשאול את הלקוח אם הוא רוצה לדבר עם אדם

4. **עיבוד נתונים**
   - בקשה לאנשים לפתור רשומות נתונים לא ברורות
   - אישור פרשנות AI על מסמכים לא ברורים
   - לאפשר למשתמשים לבחור בין פרשנויות תקפות מרובות

5. **מערכות קריטיות לבטיחות**
   - דרישת אישור אדם לפני פעולות בלתי הפיכות
   - קבלת אישור לפני גישה לנתונים רגישים
   - אישור החלטות בתעשיות מפוקחות (בריאות, פיננסים)

6. **סוכנים אינטראקטיביים**
   - בניית בוטים שיחותיים שמבקשים שאלות המשך
   - יצירת ויזרדים שמדריכים משתמשים בתהליכים מורכבים
   - עיצוב סוכנים שמשתפים פעולה עם אנשים צעד אחרי צעד

### 🔄 השוואה: עם אדם בלולאה מול ללא

| תכונה | תהליך עבודה מותנה | תהליך עבודה עם אדם בלולאה |
|---------|---------------------|---------------------------|
| **ביצוע** | `workflow.run()` יחיד | לולאה עם `run_stream()` + `send_responses_streaming()` |
| **קלט משתמש** | ללא (מלא אוטומטי) | בקשות אינטראקטיביות דרך `input()` או UI |
| **רכיבים** | סוכנים + מבצעים | + RequestInfoExecutor + DecisionManager |
| **אירועים** | רק AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, ועוד |
| **השהייה** | אין השהייה | תהליך העבודה נעצר ב-RequestInfoExecutor |
| **שליטה אנושית** | ללא שליטה אנושית | אנשים מקבלים החלטות מרכזיות |
| **מקרה שימוש** | אוטומציה | שיתוף פעולה ופיקוח |

### 🚀 תבניות מתקדמות:

#### נקודות החלטה רבות של אדם
ניתן לכלול כמה צמתים של `RequestInfoExecutor` באותו תהליך עבודה:
```python
.add_edge(agent1, request_info_1)  # החלטת אדם ראשונה
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # החלטת אדם שנייה
.add_edge(decision_manager_2, final_agent)
```

#### טיפול בזמן תפוגה
יישום טיימאאוט לתגובות אנושיות:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # ברירת מחדל לאופציה הבטוחה
```

#### אינטגרציה עשירה עם UI
במקום `input()`, אינטגרציה עם UI ווב, Slack, Teams וכו':
```python
if isinstance(event, RequestInfoEvent):
    # שלח התראה לערוץ המועדף על המשתמש
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### אדם בלולאה מותנה
בקשת קלט אדם רק במצבים ספציפיים:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # העבר רק לאדם אם הביטחון נמוך או שהערך גבוה
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ פרקטיקות מומלצות:

1. **תמיד לגשת ל-RequestInfoMessage כמחלקת בסיס**
   - מספק בטיחות טיפוס ואימות
   - מאפשר הקשר עשיר להצגת UI
   - מבהיר את הכוונה של כל סוג בקשה

2. **השתמש בפקודות תיאוריות**
   - כלול הקשר לגבי מה שאתה שואל
   - הסבר את ההשלכות של כל בחירה
   - שמור על שאלות פשוטות וברורות

3. **נהל קלט לא צפוי**
   - אמת תגובות משתמש
   - ספק ברירות מחדל לקלט לא תקין
   - תן הודעות שגיאה ברורות

4. **עקוב אחר מזהי בקשות**
   - נצל את הקורלציה בין request_id לתגובות
   - אל תנסה לנהל מצב ידנית

5. **עיצוב ללא חסימה**
   - אל תחסום תהליכים בהמתנה לקלט
   - השתמש בתבניות אסינכרוניות בכל התהליך
   - תמוך בהרצת מופעי תהליך מקבילים

### 📚 מושגים קשורים:

- **תיווך סוכן** - חטיפת קריאות סוכן (פנקס קודמות)
- **ניהול מצב תהליך עבודה** - שמירת מצב בין הרצות
- **שיתוף פעולה רב-סוכני** - שילוב אדם בלולאה עם צוותי סוכנים
- **ארכיטקטורות מונעות אירועים** - בניית מערכות תגובתיות עם אירועים

---

### 🎓 מזל טוב!

שלטת בתהליכי אדם בלולאה עם Microsoft Agent Framework! עכשיו אתה יודע כיצד:
- ✅ לעצור תהליכי עבודה לאיסוף קלט אדם
- ✅ להשתמש ב-RequestInfoExecutor ו-RequestInfoMessage
- ✅ לטפל בביצוע זרימה עם אירועים
- ✅ ליצור מבצעים מותאמים אישית עם @handler
- ✅ לנתב תהליכי עבודה על בסיס החלטות אדם
- ✅ לבנות סוכני AI אינטראקטיביים שמשתפים פעולה עם אנשים

**זו תבנית קריטית לבניית מערכות AI אמינות וניתנות לשליטה!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
